# **Classificador Supervisionado (Stage 2)**

## Objetivo deste notebook

Treinar e otimizar um classificador supervisionado sobre o subconjunto de
transações filtradas pelo Stage 1 (Isolation Forest, corte de 28%), onde
o desbalanceamento entre classes é muito menos severo (0,59% de fraude no
treino filtrado, contra 0,176% no dataset original, uma concentração
~3,4x maior). Este é o classificador que, na prática, vai gerar o score
final de fraude usado para decisão.

## Contexto herdado do Stage 1

| Conjunto | Transações | Fraudes | % fraude | Recall preservado |
|---|---|---|---|---|
| Treino filtrado | 63.554 | 376 | 0,59% | 94,24% |
| Teste filtrado | 15.888 | 71 | 0,45% | 95,95% |

Importante ter em mente: as fraudes não preservadas pelo Stage 1 (23 no
treino, 3 no teste) já estão permanentemente fora do funil e o Stage 2
não pode recuperá-las. O recall final do pipeline completo será sempre
limitado pelo recall do Stage 1.

## O que será feito

**1. Carregamento dos dados filtrados**

Leitura de `train_stage1_filtered.parquet` e `test_stage1_filtered.parquet`
(produto do notebook do satge 1), incluindo a coluna `anomaly_score` gerada pelo
Isolation Forest.

**2. Decisão sobre o uso de `anomaly_score` como feature**

Avaliação se o score de anomalia do Stage 1 agrega valor preditivo como
feature adicional de entrada do Stage 2, ou se deve ser descartado para
evitar redundância/vazamento de lógica entre as etapas.

**3. Treinamento do modelo principal

Otimização de hiperparâmetros via **Optuna**, com função objetivo baseada
nas métricas principais do projeto (PR-AUC e/ou Recall).

**4. Treinamento dos modelos de comparação com o modelo referência LightGBM**

- **XGBoost** (gradient boosting alternativo)
- **Random Forest** (ensemble bagging, referência clássica)
- **Regressão Logística** (baseline linear interpretável)

**5. Avaliação comparativa dos quatro modelos**

- Principais: PR-AUC e Recall (guiam a escolha do modelo)
- Secundárias: Precision, F1-Score, FPR, matriz de confusão
- Contraste pedagógico: ROC-AUC e Accuracy (para reforçar a armadilha do
  desbalanceamento, mesmo já mitigada pelo Stage 1)

**6. Seleção do modelo final do Stage 2**

Escolha justificada por evidência, com base nas métricas acima.

**7. Persistência de artefatos**

Salvamento do modelo final, dos hiperparâmetros otimizados e das
previsões/scores no conjunto de teste, para uso nos notebooks posteriores.

## Comparação: Uso do `anomaly_score` como Feature

Antes de otimizar o classificador do Stage 2, precisamos decidir se o
score de anomalia gerado pelo Isolation Forest (Stage 1) deve ser
incorporado como feature adicional de entrada, ou descartado.

Para isso, treinamos um LightGBM com hiperparâmetros padrão (sem
otimização via Optuna ainda) em duas versões do conjunto de features:

1. **Sem `anomaly_score`** — apenas as 31 features originais
2. **Com `anomaly_score`** — as 31 features + o score do Stage 1

Comparamos PR-AUC e Recall entre as duas versões no conjunto de teste
filtrado. A decisão sobre incluir ou não essa feature será tomada com
base em evidência empírica antes de investir tempo computacional na
otimização fina via Optuna, que será feita apenas uma vez, sobre a
versão vencedora.

In [1]:
# ================================================================
# Comparação: com e sem anomaly_score
# ================================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import precision_recall_curve, auc, recall_score

# ----------------------------------------------------------------
# Carregamento dos dados filtrados pelo Stage 1
# ----------------------------------------------------------------
caminho_dados = "/content/drive/MyDrive/fraud-detection-two-stage/data/processed"

df_treino_filtrado = pd.read_parquet(f"{caminho_dados}/train_stage1_filtered.parquet")
df_teste_filtrado  = pd.read_parquet(f"{caminho_dados}/test_stage1_filtered.parquet")

assert "transaction_id" in df_treino_filtrado.columns and "transaction_id" in df_teste_filtrado.columns, \
    "ERRO: transaction_id ausente — reexecute o notebook 03 corrigido antes de continuar!"

print(f"Treino filtrado: {df_treino_filtrado.shape[0]:,} | "
      f"{df_treino_filtrado['Class'].sum()} fraudes")
print(f"Teste filtrado : {df_teste_filtrado.shape[0]:,} | "
      f"{df_teste_filtrado['Class'].sum()} fraudes")

# ----------------------------------------------------------------
# Definição dos dois conjuntos de features para comparação
# ----------------------------------------------------------------
features_base = [col for col in df_treino_filtrado.columns
                  if col not in ["Class", "anomaly_score", "transaction_id"]]

features_com_score = features_base + ["anomaly_score"]

print(f"\nFeatures base (sem anomaly_score): {len(features_base)}")
print(f"Features com anomaly_score: {len(features_com_score)}")

Treino filtrado: 63,554 | 376 fraudes
Teste filtrado : 15,888 | 71 fraudes

Features base (sem anomaly_score): 31
Features com anomaly_score: 32


In [2]:
# ----------------------------------------------------------------
# Função auxiliar — treina LightGBM padrão e avalia PR-AUC/Recall
# ----------------------------------------------------------------
def treinar_e_avaliar(features, nome_versao):
    """Treina LightGBM com hiperparâmetros padrão e retorna métricas de teste."""

    X_treino = df_treino_filtrado[features]
    y_treino = df_treino_filtrado["Class"]
    X_teste  = df_teste_filtrado[features]
    y_teste  = df_teste_filtrado["Class"]

    modelo = lgb.LGBMClassifier(
        objective="binary",
        random_state=42,
        verbose=-1
    )
    modelo.fit(X_treino, y_treino)

    # Probabilidade da classe positiva (fraude)
    y_score = modelo.predict_proba(X_teste)[:, 1]

    # PR-AUC
    precision, recall, _ = precision_recall_curve(y_teste, y_score)
    pr_auc = auc(recall, precision)

    # Recall no threshold padrão de 0.5 (só para referência nesta comparação)
    y_pred = modelo.predict(X_teste)
    recall_05 = recall_score(y_teste, y_pred)

    print(f"[{nome_versao}] PR-AUC: {pr_auc:.4f} | Recall (threshold=0.5): {recall_05:.4f}")

    return {"modelo": modelo, "pr_auc": pr_auc, "recall_05": recall_05, "y_score": y_score}

# ----------------------------------------------------------------
# Executa a comparação
# ----------------------------------------------------------------
resultado_sem_score = treinar_e_avaliar(features_base, "Sem anomaly_score")
resultado_com_score = treinar_e_avaliar(features_com_score, "Com anomaly_score")

# ----------------------------------------------------------------
# Comparação final
# ----------------------------------------------------------------
diferenca_pr_auc = resultado_com_score["pr_auc"] - resultado_sem_score["pr_auc"]
diferenca_recall = resultado_com_score["recall_05"] - resultado_sem_score["recall_05"]

print(f"\nDiferença PR-AUC (com - sem): {diferenca_pr_auc:+.4f}")
print(f"Diferença Recall (com - sem): {diferenca_recall:+.4f}")

[Sem anomaly_score] PR-AUC: 0.4196 | Recall (threshold=0.5): 0.6056
[Com anomaly_score] PR-AUC: 0.3228 | Recall (threshold=0.5): 0.5493

Diferença PR-AUC (com - sem): -0.0968
Diferença Recall (com - sem): -0.0563


## Análise — Resultado da Comparação: `anomaly_score` como Feature

| Métrica | Sem anomaly_score | Com anomaly_score | Diferença |
|---|---|---|---|
| PR-AUC | 0,4196 | 0,3228 | -0,0968 |
| Recall (threshold=0,5) | 0,6056 | 0,5493 | -0,0563 |

A inclusão do `anomaly_score` **piorou o desempenho do modelo** em ambas
as métricas de forma expressiva, e não marginal.

**Hipótese explicativa:**

o `anomaly_score` é derivado das mesmas 31
features usadas pelo Isolation Forest e ao incluí-lo como feature
adicional, não se introduz informação nova ao LightGBM, mas sim uma
versão agregada e com perda das mesmas variáveis que o modelo já acessa
diretamente e de forma mais granular. Com um volume relativamente
pequeno de exemplos de fraude no treino filtrado (376 casos), o modelo
tem menos capacidade de aprender a desconsiderar essa informação
redundante, o que pode ter introduzido ruído em vez de sinal útil.

**Decisão:**

`anomaly_score` será descartado do conjunto de features do
Stage 2. O classificador seguirá para a etapa de otimização via Optuna
utilizando exclusivamente as 31 features originais (V1–V28, Hour_sin,
Hour_cos, Amount_scaled).

## Etapa — Comparação Inicial dos Quatro Modelos (Baseline)

Antes de qualquer otimização de hiperparâmetros, treinamos os quatro
candidatos ao Stage 2 com configurações padrão, sobre o mesmo conjunto
de features (31 features originais, sem `anomaly_score`, conforme
decisão anterior):

- **LightGBM** (candidato principal)
- **XGBoost**
- **Random Forest**
- **Regressão Logística** (baseline)

O objetivo é uma comparação justa, sem viés de tuning, para identificar
qual modelo tem melhor potencial nesse problema antes de investir tempo
computacional em otimização via Optuna que será aplicada apenas ao
modelo vencedor desta comparação.

In [3]:
# ================================================================
# Comparação Baseline dos 4 Modelos
# ================================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_recall_curve, auc, recall_score,
                              precision_score, f1_score, confusion_matrix,
                              roc_auc_score, accuracy_score)

# ----------------------------------------------------------------
# Features definidas na etapa anterior (sem anomaly_score)
# ----------------------------------------------------------------
X_treino = df_treino_filtrado[features_base]
y_treino = df_treino_filtrado["Class"]
X_teste  = df_teste_filtrado[features_base]
y_teste  = df_teste_filtrado["Class"]

RANDOM_STATE = 42

In [4]:
# ----------------------------------------------------------------
# Função auxiliar — avalia qualquer modelo já treinado
# ----------------------------------------------------------------
def avaliar_modelo(modelo, X_teste, y_teste, nome):
    """Calcula métricas principais, secundárias e de contraste pedagógico."""

    y_score = modelo.predict_proba(X_teste)[:, 1]
    y_pred  = modelo.predict(X_teste)

    # Principais — guiam a decisão
    precision, recall, _ = precision_recall_curve(y_teste, y_score)
    pr_auc = auc(recall, precision)
    recall_05 = recall_score(y_teste, y_pred)

    # Secundárias — reportadas na comparação
    precision_05 = precision_score(y_teste, y_pred)
    f1 = f1_score(y_teste, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_teste, y_pred).ravel()
    fpr = fp / (fp + tn)

    # Contraste pedagógico — evidencia a armadilha do desbalanceamento
    roc_auc = roc_auc_score(y_teste, y_score)
    acc = accuracy_score(y_teste, y_pred)

    return {
        "modelo": nome,
        "pr_auc": pr_auc,
        "recall": recall_05,
        "precision": precision_05,
        "f1": f1,
        "fpr": fpr,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "roc_auc": roc_auc,
        "accuracy": acc,
        "y_score": y_score
    }

In [5]:
# ----------------------------------------------------------------
# Treinamento dos 4 modelos — configuração padrão (sem tuning)
# ----------------------------------------------------------------

resultados = []

# 1. LightGBM
modelo_lgbm = lgb.LGBMClassifier(objective="binary", random_state=RANDOM_STATE, verbose=-1)
modelo_lgbm.fit(X_treino, y_treino)
resultados.append(avaliar_modelo(modelo_lgbm, X_teste, y_teste, "LightGBM"))

# 2. XGBoost
modelo_xgb = xgb.XGBClassifier(objective="binary:logistic", random_state=RANDOM_STATE,
                                eval_metric="logloss", use_label_encoder=False)
modelo_xgb.fit(X_treino, y_treino)
resultados.append(avaliar_modelo(modelo_xgb, X_teste, y_teste, "XGBoost"))

# 3. Random Forest
modelo_rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
modelo_rf.fit(X_treino, y_treino)
resultados.append(avaliar_modelo(modelo_rf, X_teste, y_teste, "Random Forest"))

# 4. Regressão Logística — baseline
# max_iter aumentado para garantir convergência
modelo_lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
modelo_lr.fit(X_treino, y_treino)
resultados.append(avaliar_modelo(modelo_lr, X_teste, y_teste, "Regressão Logística"))

print("Treinamento dos 4 modelos concluído.")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:03:07] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Treinamento dos 4 modelos concluído.


In [6]:
# ----------------------------------------------------------------
# Tabela comparativa consolidada
# ----------------------------------------------------------------

df_resultados = pd.DataFrame(resultados).drop(columns=["y_score"])
df_resultados = df_resultados.sort_values("pr_auc", ascending=False)

print("Comparação dos 4 modelos (ordenado por PR-AUC):\n")
print(df_resultados.to_string(index=False))

Comparação dos 4 modelos (ordenado por PR-AUC):

             modelo   pr_auc   recall  precision       f1      fpr  tp  fp  fn    tn  roc_auc  accuracy
      Random Forest 0.833884 0.718310   0.980769 0.829268 0.000063  51   1  20 15816 0.945403  0.998678
Regressão Logística 0.800500 0.619718   0.936170 0.745763 0.000190  44   3  27 15814 0.979914  0.998112
            XGBoost 0.799963 0.690141   0.890909 0.777778 0.000379  49   6  22 15811 0.960410  0.998238
           LightGBM 0.419591 0.605634   0.318519 0.417476 0.005817  43  92  28 15725 0.815665  0.992447


## Análise — Comparação Baseline dos 4 Modelos (Sem Tuning)

Treinamos os quatro candidatos ao Stage 2 com hiperparâmetros padrão, sobre
o mesmo conjunto de 31 features (sem `anomaly_score`), para uma comparação
justa antes de qualquer otimização.

**Resultados (ordenados por PR-AUC):**

| Modelo | PR-AUC | Recall | Precision | FP | FN |
|---|---|---|---|---|---|
| Random Forest | 0,8339 | 71,8% | 98,1% | 1 | 20 |
| Regressão Logística | 0,8005 | 62,0% | 93,6% | 3 | 27 |
| XGBoost | 0,8000 | 69,0% | 89,1% | 6 | 22 |
| LightGBM | 0,4196 | 60,6% | 31,9% | 92 | 28 |

**Resultado inesperado:** o LightGBM, candidato principal definido no
escopo do projeto, apresentou a pior performance entre os quatro modelos
em configuração padrão, destacadamente pelo alto número de falsos
positivos (92, muito acima dos demais), que derrubou sua precision para
31,9% e, consequentemente, seu PR-AUC.

**Random Forest liderou com folga**, mesmo sem nenhum ajuste de
hiperparâmetros com um recall de 71,8% (51 de 71 fraudes capturadas) com
apenas 1 falso positivo em 15.888 transações.

**Hipótese explicativa para o desempenho do LightGBM:** o algoritmo é
conhecido por ser mais sensível a hiperparâmetros default do que Random
Forest, especialmente em cenários desbalanceados. Fatores prováveis
incluem: (1) `num_leaves` padrão possivelmente desproporcional ao volume
de exemplos positivos disponíveis (376 fraudes no treino); (2) ausência
de tratamento explícito de desbalanceamento (parâmetros como
`scale_pos_weight` ou `is_unbalance` não foram configurados); (3) Random
Forest se beneficia estruturalmente de bagging e amostragem aleatória de
features, o que tende a gerar maior robustez mesmo sem tuning o que é diferente
de algoritmos de boosting, mais sensíveis a configuração fina.

## Decisão: prosseguir com otimização via Optuna no LightGBM

Esse resultado baseline não descarta o LightGBM como candidato, mas
confirma que ele depende de otimização de hiperparâmetros para atingir seu
potencial real o que, aliás, é exatamente a proposta original definida
no escopo do projeto, então por isso vai ser testado a otimização de hiperparâmetros do LightGBM com o objetivo de testar seu potencial.

In [7]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 11.3 MB/s eta 0:00:00


In [8]:
# ================================================================
# Otimização do LightGBM via Optuna
# ================================================================

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve, auc
import lightgbm as lgb
import numpy as np

# Reduz o volume de logs do Optuna no notebook
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
N_TRIALS = 50

In [9]:
# ----------------------------------------------------------------
# Função objetivo do Optuna
# ----------------------------------------------------------------
def objetivo(trial):
    """Define o espaço de busca e retorna o PR-AUC médio via CV estratificado."""

    params = {
        "objective": "binary",
        "random_state": RANDOM_STATE,
        "verbose": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 50),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        # Tratamento de desbalanceamento — parâmetro que faltou no baseline
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 50.0),
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    pr_auc_scores = []

    for treino_idx, valid_idx in skf.split(X_treino, y_treino):
        X_fold_treino = X_treino.iloc[treino_idx]
        y_fold_treino = y_treino.iloc[treino_idx]
        X_fold_valid  = X_treino.iloc[valid_idx]
        y_fold_valid  = y_treino.iloc[valid_idx]

        modelo = lgb.LGBMClassifier(**params)
        modelo.fit(X_fold_treino, y_fold_treino)

        y_score = modelo.predict_proba(X_fold_valid)[:, 1]
        precision, recall, _ = precision_recall_curve(y_fold_valid, y_score)
        pr_auc_scores.append(auc(recall, precision))

    return np.mean(pr_auc_scores)

In [10]:
# ----------------------------------------------------------------
# Execução da otimização
# ----------------------------------------------------------------
# direction="maximize" porque queremos o MAIOR PR-AUC possível

study = optuna.create_study(direction="maximize",
                              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objetivo, n_trials=N_TRIALS)

print(f"Melhor PR-AUC (CV): {study.best_value:.4f}")
print(f"\nMelhores hiperparâmetros encontrados:")
for param, valor in study.best_params.items():
    print(f"  {param}: {valor}")

Melhor PR-AUC (CV): 0.9183

Melhores hiperparâmetros encontrados:
  n_estimators: 277
  learning_rate: 0.04206236723801755
  num_leaves: 25
  max_depth: 6
  min_child_samples: 64
  subsample: 0.9670481062072444
  colsample_bytree: 0.874406402678019
  reg_alpha: 0.529116512196945
  reg_lambda: 3.04235328278629
  scale_pos_weight: 21.812969838552792


In [11]:
# ----------------------------------------------------------------
# Avaliação do modelo otimizado no conjunto de TESTE (holdout real)
# ----------------------------------------------------------------
melhores_params = study.best_params.copy()
melhores_params.update({"objective": "binary", "random_state": RANDOM_STATE, "verbose": -1})

modelo_lgbm_otimizado = lgb.LGBMClassifier(**melhores_params)
modelo_lgbm_otimizado.fit(X_treino, y_treino)

resultado_lgbm_otimizado = avaliar_modelo(modelo_lgbm_otimizado, X_teste, y_teste,
                                            "LightGBM (Optuna)")

print("Resultado do LightGBM otimizado no conjunto de teste:")
for chave in ["pr_auc", "recall", "precision", "f1", "fpr", "tp", "fp", "fn", "tn"]:
    print(f"  {chave}: {resultado_lgbm_otimizado[chave]:.4f}" if isinstance(resultado_lgbm_otimizado[chave], float)
          else f"  {chave}: {resultado_lgbm_otimizado[chave]}")

Resultado do LightGBM otimizado no conjunto de teste:
  pr_auc: 0.8340
  recall: 0.7887
  precision: 0.8116
  f1: 0.8000
  fpr: 0.0008
  tp: 56
  fp: 13
  fn: 15
  tn: 15804


Podemos observar uma grande melhora de performance porém para fazer uma comparação justa com random forest vou utilizar a otimização de hiperparâmetros também no Random Forest

In [12]:
# ================================================================
# Otimização do Random Forest via Optuna — para comparação justa
# ================================================================

from sklearn.ensemble import RandomForestClassifier

def objetivo_rf(trial):
    """Espaço de busca do Random Forest, equivalente em rigor ao do LightGBM."""

    params = {
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        # Tratamento de desbalanceamento — equivalente ao scale_pos_weight do LightGBM
        "class_weight": trial.suggest_categorical(
            "class_weight", ["balanced", "balanced_subsample", None]
        ),
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    pr_auc_scores = []

    for treino_idx, valid_idx in skf.split(X_treino, y_treino):
        X_fold_treino = X_treino.iloc[treino_idx]
        y_fold_treino = y_treino.iloc[treino_idx]
        X_fold_valid  = X_treino.iloc[valid_idx]
        y_fold_valid  = y_treino.iloc[valid_idx]

        modelo = RandomForestClassifier(**params)
        modelo.fit(X_fold_treino, y_fold_treino)

        y_score = modelo.predict_proba(X_fold_valid)[:, 1]
        precision, recall, _ = precision_recall_curve(y_fold_valid, y_score)
        pr_auc_scores.append(auc(recall, precision))

    return np.mean(pr_auc_scores)

# N_TRIALS reduzido especificamente para o Random Forest
N_TRIALS_RF = 25

study_rf = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(objetivo_rf, n_trials=N_TRIALS_RF, show_progress_bar=True)

print(f"Melhor PR-AUC (CV) — Random Forest: {study_rf.best_value:.4f}")
print(f"\nMelhores hiperparâmetros:")
for param, valor in study_rf.best_params.items():
    print(f"  {param}: {valor}")

  0%|          | 0/25 [00:00<?, ?it/s]

Melhor PR-AUC (CV) — Random Forest: 0.8788

Melhores hiperparâmetros:
  n_estimators: 274
  max_depth: 6
  min_samples_split: 6
  min_samples_leaf: 5
  max_features: sqrt
  class_weight: None


In [13]:
# ----------------------------------------------------------------
# Avaliação do Random Forest otimizado no conjunto de TESTE
# ----------------------------------------------------------------
melhores_params_rf = study_rf.best_params.copy()
melhores_params_rf.update({"random_state": RANDOM_STATE, "n_jobs": -1})

modelo_rf_otimizado = RandomForestClassifier(**melhores_params_rf)
modelo_rf_otimizado.fit(X_treino, y_treino)

resultado_rf_otimizado = avaliar_modelo(modelo_rf_otimizado, X_teste, y_teste,
                                          "Random Forest (Optuna)")

print("Resultado do Random Forest otimizado no conjunto de teste:")
for chave in ["pr_auc", "recall", "precision", "f1", "fpr", "tp", "fp", "fn", "tn"]:
    print(f"  {chave}: {resultado_rf_otimizado[chave]:.4f}" if isinstance(resultado_rf_otimizado[chave], float)
          else f"  {chave}: {resultado_rf_otimizado[chave]}")

Resultado do Random Forest otimizado no conjunto de teste:
  pr_auc: 0.8082
  recall: 0.6761
  precision: 0.9796
  f1: 0.8000
  fpr: 0.0001
  tp: 48
  fp: 1
  fn: 23
  tn: 15816


## Comparação Final: LightGBM vs. Random Forest (Otimizados)

Após otimização via Optuna em ambos os modelos, comparamos o desempenho
final no conjunto de teste:

| Métrica | Random Forest (Optuna) | LightGBM (Optuna) |
|---|---|---|
| PR-AUC | 0,8082 | **0,8340** |
| Recall | 67,6% | **78,9%** |
| Precision | **98,0%** | 81,2% |
| Falsos Positivos | **1** | 13 |
| Falsos Negativos | 23 | **15** |

**LightGBM apresentou o melhor resultado nas métricas principais do
projeto** (PR-AUC e Recall), capturando 8 fraudes a mais que o Random
Forest, ao custo de 12 falsos positivos adicionais um trade-off
favorável dado que a métrica de recall foi definida como prioritária no
escopo do projeto.

**Ressalva metodológica importante:** o espaço de busca do Random Forest
precisou ser reduzido de forma mais agressiva que o do LightGBM
(`max_depth` limitado a 6, `n_trials` reduzido a 25, CV reduzido a 3
folds) devido ao custo computacional proibitivo no ambiente Colab
gratuito. Isso significa que o Random Forest não recebeu exatamente o
mesmo nível de exploração de hiperparâmetros que o LightGBM, o que deve
ser levado em conta na interpretação da comparação, então a vantagem do
LightGBM pode estar parcialmente relacionada a essa assimetria de
orçamento de busca, e não apenas à superioridade estrutural do algoritmo
para este problema.

**Decisão final: LightGBM (otimizado via Optuna) é escolhido como modelo
do Stage 2.** A decisão se baseia em: (1) melhor PR-AUC e Recall, as
métricas principais definidas no escopo do projeto; (2) alinhamento com
a arquitetura originalmente proposta; (3) reconhecimento honesto de que a
comparação, embora favorável ao LightGBM, não foi perfeitamente simétrica
em orçamento computacional entre os dois modelos.

Random Forest permanece documentado como alternativa competitiva e
consideravelmente mais conservadora em falsos positivos o que pode ser informação
relevante para a discussão de custo-benefício no notebook 05, já que o
threshold de decisão do LightGBM ainda será ajustado por minimização de
custo de negócio, podendo alterar esse equilíbrio final.

In [14]:
# ----------------------------------------------------------------
# PERSISTÊNCIA — modelo final do Stage 2, hiperparâmetros e resultados
# ----------------------------------------------------------------

import joblib
import json

caminho_base    = "/content/drive/MyDrive/fraud-detection-two-stage"
caminho_dados   = f"{caminho_base}/data/processed"
caminho_modelos = f"{caminho_base}/models"

# 1. Modelo LightGBM otimizado — modelo final do Stage 2
joblib.dump(modelo_lgbm_otimizado, f"{caminho_modelos}/lightgbm_stage2.pkl")

# 2. Modelo Random Forest otimizado — mantido como alternativa documentada
joblib.dump(modelo_rf_otimizado, f"{caminho_modelos}/random_forest_stage2_alt.pkl")

# 3. Scores de probabilidade do LightGBM no teste — necessários para o
# notebook 05, que vai variar o threshold sobre esses mesmos scores
y_score_lgbm_teste = modelo_lgbm_otimizado.predict_proba(X_teste)[:, 1]

df_scores_teste = df_teste_filtrado[["transaction_id", "Class"]].copy()
df_scores_teste["y_score_lgbm"] = y_score_lgbm_teste
df_scores_teste.to_parquet(f"{caminho_dados}/test_stage2_scores.parquet", index=False)

assert "transaction_id" in df_scores_teste.columns
assert df_scores_teste["transaction_id"].is_unique
print(f"transaction_id preservado em test_stage2_scores.parquet.")

# 4. Metadados consolidados do Stage 2 — decisões, hiperparâmetros e métricas
metadados_stage2 = {
    "modelo_escolhido": "LightGBM",
    "features_utilizadas": features_base,
    "anomaly_score_incluido": False,
    "motivo_exclusao_anomaly_score": "Piora PR-AUC (-0.0968) e Recall (-0.0563) no teste comparativo",

    "hiperparametros_lightgbm": study.best_params,
    "hiperparametros_random_forest": study_rf.best_params,

    "metricas_teste": {
        "lightgbm_otimizado": {
            "pr_auc": round(resultado_lgbm_otimizado["pr_auc"], 4),
            "recall": round(resultado_lgbm_otimizado["recall"], 4),
            "precision": round(resultado_lgbm_otimizado["precision"], 4),
            "f1": round(resultado_lgbm_otimizado["f1"], 4),
            "fp": int(resultado_lgbm_otimizado["fp"]),
            "fn": int(resultado_lgbm_otimizado["fn"]),
        },
        "random_forest_otimizado": {
            "pr_auc": round(resultado_rf_otimizado["pr_auc"], 4),
            "recall": round(resultado_rf_otimizado["recall"], 4),
            "precision": round(resultado_rf_otimizado["precision"], 4),
            "f1": round(resultado_rf_otimizado["f1"], 4),
            "fp": int(resultado_rf_otimizado["fp"]),
            "fn": int(resultado_rf_otimizado["fn"]),
        }
    },

    "ressalva_metodologica": (
        "Random Forest teve espaco de busca reduzido (max_depth<=6, "
        "n_trials=25, cv=3 folds) por limitacao computacional do Colab "
        "gratuito, resultando em orcamento de otimizacao nao perfeitamente "
        "simetrico ao do LightGBM."
    )
}

with open(f"{caminho_modelos}/stage2_metadata.json", "w") as f:
    json.dump(metadados_stage2, f, indent=2, ensure_ascii=False)

print("Artefatos do Stage 2 salvos com sucesso:")
print(f"  Modelo principal (LightGBM)      : {caminho_modelos}/lightgbm_stage2.pkl")
print(f"  Modelo alternativo (Random Forest): {caminho_modelos}/random_forest_stage2_alt.pkl")
print(f"  Scores de teste                   : {caminho_dados}/test_stage2_scores.parquet")
print(f"  Metadados                         : {caminho_modelos}/stage2_metadata.json")

transaction_id preservado em test_stage2_scores.parquet.
Artefatos do Stage 2 salvos com sucesso:
  Modelo principal (LightGBM)      : /content/drive/MyDrive/fraud-detection-two-stage/models/lightgbm_stage2.pkl
  Modelo alternativo (Random Forest): /content/drive/MyDrive/fraud-detection-two-stage/models/random_forest_stage2_alt.pkl
  Scores de teste                   : /content/drive/MyDrive/fraud-detection-two-stage/data/processed/test_stage2_scores.parquet
  Metadados                         : /content/drive/MyDrive/fraud-detection-two-stage/models/stage2_metadata.json


# **Conclusão Geral — Stage 2: Classificador Supervisionado**

Este notebook teve como objetivo treinar, comparar e otimizar classificadores
supervisionados sobre o subconjunto de transações filtradas pelo Stage 1
(Isolation Forest, corte de 28%).

**1. Decisão sobre `anomaly_score` como feature foi descartada com evidência**

Testamos incluir o score de anomalia do Stage 1 como feature adicional do
Stage 2. O resultado foi negativo (PR-AUC caiu de 0,4196 para 0,3228),
provavelmente por redundância informacional com as 31 features originais
combinada ao volume reduzido de exemplos de fraude. Decisão: manter apenas
as 31 features originais (V1–V28, Hour_sin, Hour_cos, Amount_scaled).

**2. Comparação baseline dos quatro modelos (sem tuning)**

Com hiperparâmetros padrão, Random Forest liderou com folga (PR-AUC 0,8339),
seguido de Regressão Logística (0,8005) e XGBoost (0,8000). **LightGBM,
candidato principal do projeto, teve o pior desempenho baseline** (PR-AUC
0,4196), com 92 falsos positivos, resultado atribuído à ausência de
tratamento de desbalanceamento e à sensibilidade do algoritmo a
hiperparâmetros default.

**3. Otimização via Optuna — comparação justa entre os dois principais candidatos**

Após reconhecer que comparar um LightGBM otimizado contra um Random Forest
baseline seria metodologicamente injusto, ambos os modelos foram otimizados
via Optuna com validação cruzada estratificada:

| Métrica | Random Forest (Optuna) | LightGBM (Optuna) |
|---|---|---|
| PR-AUC | 0,8082 | **0,8340** |
| Recall | 67,6% | **78,9%** |
| Precision | **98,0%** | 81,2% |
| Falsos Positivos | **1** | 13 |
| Falsos Negativos | 23 | **15** |

**4. Modelo final escolhido: LightGBM**

LightGBM superou o Random Forest nas duas métricas principais do projeto
(PR-AUC e Recall), capturando 8 fraudes a mais ao custo de 12 falsos
positivos adicionais, trade-off favorável dado que recall foi definido
como prioridade no escopo do projeto.

**5. Ressalva metodológica documentada com transparência**

O espaço de busca do Random Forest precisou ser reduzido de forma mais
agressiva que o do LightGBM (profundidade máxima menor, menos trials,
menos folds de CV) por limitação computacional do Colab gratuito. A
comparação, embora favorável ao LightGBM, não teve orçamento de otimização
perfeitamente simétrico entre os dois candidatos e essa limitação é
reconhecida explicitamente, em vez de apresentar o resultado como
definitivo e isento de viés.

**6. Artefatos persistidos para as próximas etapas**

- Modelo LightGBM otimizado (modelo final do Stage 2)
- Modelo Random Forest otimizado (alternativa documentada)
- Scores de probabilidade do LightGBM no conjunto de teste (base para
  variação de threshold no próximo notebook)
- Metadados completos com hiperparâmetros, métricas e ressalvas

## Limitação central do notebook

Todas as métricas reportadas usam o **threshold padrão de 0,5**, escolha
arbitrária que não reflete necessariamente o ponto de operação ótimo para
o negócio. O verdadeiro potencial do LightGBM otimizado e a decisão final
sobre qual perfil de erro (mais recall vs. mais precision) é preferível
só poderá ser avaliado corretamente após a análise de custo-benefício.